# Preprocessing Text For Word Embeddings

In [ ]:
import pandas as pd

# Bert imported necessary libraries
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# !pip install fasttext
# import fasttext

# import fasttext.util
# fasttext.util.download_model('en', if_exists='ignore')  # English
# ft = fasttext.load_model('cc.en.300.bin')

training = pd.read_csv("sent_train.csv")

KeyboardInterrupt: 

In [ ]:
# Set a random seed
random_seed = 42
random.seed(random_seed)

# Set a random seed for PyTorch (for GPU as well)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

In [ ]:
# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
from google.colab import files
#files.upload()

!pip install emoji nltk

import re, emoji, nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

valid = pd.read_csv("sent_valid.csv")
print("Columns:", training.columns.tolist())



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.8 MB/s eta 0:00:00
Train shape: (9543, 2)  Valid shape: (2388, 2)
Columns: ['text', 'label']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
display(training.head(5))

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [ ]:

# Load the dataset
training = pd.read_csv("/content/sent_train.csv")
valid = pd.read_csv("/content/sent_valid.csv")


Shape: (9543, 2)

Column names: Index(['text', 'label'], dtype='object')

Missing values:
 text     0
label    0
dtype: int64

Label distribution:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


# Cleaning and Normalizing

In [ ]:
import pandas as pd
import re
import emoji
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

# Base stopwords list
stop_words = set(stopwords.words('english'))

# Keep key words needed for meaning
keep_words = {'and', 'but', 'or', 'is', 'are', 'was', 'were', 'should', 'would', 'could',
              'a', 'an', 'the', 'in', 'on', 'at', 'with', 'for', 'of', 'after', 'before', 'by', 'as'}

smart_stopwords = stop_words - keep_words

#for financial terms
custom_removals = {'via', 'eps', 'bln', 'ipo', 'fomc', 'mclr', 'usmca', 'nyse',
                   'gapping', 'afterhours', 'premarket', 'today', 'week', 'month',
                   'years', 'days', 'time', 'billion', 'million', 'trillion', 'say'}
# Hi guys, here are some other low-value words we could possibly add
# {'ppi', 'nse', 'reports', 'results', 'started', 'buy', 'target', 'looking', 'proud', 'vies',
#  'title', 'halts', 'mainland', 'little', 'bit', 'percent', 'should', 'concerned', 'historical',
#  'volatility', 'audited', 'company', 'health', 'stock', 'transaction', 'project', 'program',
#  'update', 'announcement', 'facility', 'otm', 'reit', 'big', 'winners', 'loan', 'ratio', 'good'}


def smart_clean_text(text):
    text = str(text)

    text = re.sub(r'\.\.\.|\…|more:', ' ', text)

    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\$\w+', '', text)
    text = re.sub(r'\b[A-Z]{2,5}:[A-Z]{1,5}\b', '', text)
    text = re.sub(r'\([A-Z]{1,5}\)', '', text)
    text = re.sub(r'\b[A-Z]{2,5}/[A-Z]{2,5}\b', '', text)

    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+(\.\d+)?%?', '', text)

    text = text.lower()

    text = ' '.join([word for word in text.split() if word not in smart_stopwords])

    text = ' '.join([word for word in text.split() if len(word) > 2])

    text = re.sub(r'\s+', ' ', text).strip()
    return text

# add a column that has the cleaned text to training and valid csv
training['clean_text'] = training['text'].astype(str).apply(smart_clean_text)
valid['clean_text'] = valid['text'].astype(str).apply(smart_clean_text)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,proportion
label,
2,0.647386
1,0.201509
0,0.151106


In [ ]:
valid = pd.read_csv("sent_valid.csv")

# Apply the same text cleaning function
valid['clean_text'] = valid['text'].astype(str).apply(smart_clean_text)

import nltk
from nltk import word_tokenize, pos_tag

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

def extract_features(text):
    words = word_tokenize(text)
    tagged = pos_tag(words)

    nouns = sum(1 for word, tag in tagged if tag.startswith('NN'))
    adjectives = sum(1 for word, tag in tagged if tag.startswith('JJ'))
    verbs = sum(1 for word, tag in tagged if tag.startswith('VB'))
    word_count = len(words)

    return pd.Series([nouns, adjectives, verbs, word_count])

training[['nouns', 'adjectives', 'verbs', 'word_count']] = training['clean_text'].apply(extract_features)
valid[['nouns', 'adjectives', 'verbs', 'word_count']] = valid['clean_text'].apply(extract_features)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


# Download Cleaned Datasets

In [ ]:
# training.to_csv('training_cleaned.csv', index=False)
# valid.to_csv('valid_cleaned.csv', index=False)

# from google.colab import files
# files.download('training_cleaned.csv')
# files.download('valid_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Generate Embeddings Using BERT (0.1B Parameters)**

In [ ]:
import pandas as pd
training_embed = pd.read_csv("training_embed.csv")

In [ ]:
training_embed["label"].value_counts()

,count
label,
2,6178
1,1923
0,1442


In [ ]:
print(training_embed.bert_embedding.sample(1))

8924    [ 3.27210426e-01 -4.59803849e-01  3.61305505e-...
Name: bert_embedding, dtype: object


In [ ]:
training_embed.sample(3)

,text,label,clean_text,nouns,adjectives,verbs,word_count,bert_embedding,financialBERT_embedding,llama_embedding
8453,Did Changing Sentiment Drive MDxHealth's (EBR:...,2,changing sentiment drive mdxhealth share price...,6,0,1,7,[ 2.41747379e-01 -3.22208315e-01 4.23823118e-...,[ 9.39361081e-02 -1.08240497e+00 -6.12033427e-...,"[0.007686462718993425, -0.007204249035567045, ..."
5714,VC Deals: AbbVie Joins $50M For Inflammatory D...,2,deals abbvie joins for inflammatory disease tr...,4,1,1,7,[ 6.08682074e-02 4.61948477e-03 3.86340022e-...,[ 1.46445602e-01 -4.96248662e-01 -3.15353811e-...,"[-0.03644357994198799, -0.0423160158097744, -0..."
9380,Nokia and Ericsson both advance after Barr say...,1,nokia and ericsson advance after barr says fir...,6,0,2,11,[ 2.18330517e-01 3.80555391e-02 9.07392055e-...,[-3.41635034e-03 -3.44432384e-01 -3.83144587e-...,"[-0.013515187427401543, 0.019398551434278488, ..."


In [ ]:
for n in training_embed["clean_text"]:
  print(n)

jpmorgan reels expectations beyond meat
nomura points bookings weakness carnival and royal caribbean
cemex cut credit suisse morgan weak building outlook
btig research cuts neutral
funko slides after piper jaffray cut
technipfmc downgraded berenberg but called top pick deutsche bank
loses bull
deutsche bank cuts hold
cowen cuts market perform
trendforce cuts iphone estimate after foxconn delay
moody warns harley davidson
citing aero ties wells slashes hexcel
intelsat cut market perform raymond james
compass point cuts sell
muddy waters goes short luckin coffee
mantech downgraded ahead difficult comps
oppenheimer cuts perform
mplx cut credit suisse potential dilution marathon strategic review
imperial downgrades msg networks amid sports free airwaves
piper hits the materialise sidelines
hovde group cuts market perform
new netflix bear steps
shopify loses bull
nomura instinet loses confidence extended stay america
twilio gets street low target virus risk
guggenheim minerd sees gloom ahea

In [ ]:
# BERT Embedding Function
def get_bert_embedding(text):
  # Tokenizes and encodes text
  inputs = tokenizer(text, # Input text
                     padding = True, # Ensures all sequences in a batch are padded to the same length
                     truncation = True, # Enables truncation for sequences that exceed a specified max length
                     return_tensors = 'pt', # Format of returned tensors in PyTorch should be torch.Tensor objects (required for pyTorch)
                     add_special_tokens = True # Adds specialized tokens such as CLS, SEP, and PAD
                     )

  # CLS is placed at the beginning to capture overall meaning via aggregation
  # SEP is used to separate two different sentences or to mark the end of a sentence
  # PAD is added to shorter sentences to match the length of the longer sentences

  # Generates embeddings using BERT
  with torch.no_grad():
    outputs = model(**inputs)

  # outputs.last_hidden_state (batch size, sequence_length, hidden_size)
  # batch_size: Amount of sentences fed into the model
  # sequence_length: How many tokens are in each sentence
  # hidden_size: Dimensionality of each token embedding (768 features)
  token_embeddings = outputs.last_hidden_state

  # Compute the average of all token embeddings to get a sentence level vector per sentence rather than per token
  mean_pooled = token_embeddings.mean(dim=1)
  return mean_pooled.squeeze().numpy()


In [ ]:
# Apply the embedding function row by row
training['embedding'] = training['clean_text'].apply(get_bert_embedding)

In [ ]:
training.head(20)

,text,label,clean_text,nouns,adjectives,verbs,word_count,embedding
0,$BYND - JPMorgan reels in expectations on Beyo...,0,jpmorgan reels expectations beyond meat,4,0,0,5,"[0.04343193, -0.17159984, -0.006442541, -0.012..."
1,$CCL $RCL - Nomura points to bookings weakness...,0,nomura points bookings weakness carnival and r...,6,1,0,8,"[-0.3959833, -0.30719957, 0.0661829, 0.0602318..."
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0,cemex cut credit suisse morgan weak building o...,6,2,0,8,"[-0.13003948, -0.18834782, 0.13463801, 0.21141..."
3,$ESS: BTIG Research cuts to Neutral https://t....,0,btig research cuts neutral,2,2,0,4,"[-0.14499637, -0.09896575, -0.38878652, 0.0516..."
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0,funko slides after piper jaffray cut,4,0,0,6,"[-0.09609649, -0.55792147, 0.21134444, 0.02747..."
5,$FTI - TechnipFMC downgraded at Berenberg but ...,0,technipfmc downgraded berenberg but called top...,5,1,2,9,"[-0.30310422, -0.42411697, 0.0786846, 0.188643..."
6,$GM - GM loses a bull https://t.co/tdUfG5HbXy,0,loses bull,2,0,0,2,"[-0.36897755, -0.11082789, -0.0016138963, 0.13..."
7,$GM: Deutsche Bank cuts to Hold https://t.co/7...,0,deutsche bank cuts hold,3,0,1,4,"[-0.016315749, -0.39343968, -0.30481496, 0.023..."
8,$GTT: Cowen cuts to Market Perform,0,cowen cuts market perform,4,0,0,4,"[0.09511597, -0.20545241, -0.3813022, 0.052350..."
9,$HNHAF $HNHPD $AAPL - Trendforce cuts iPhone e...,0,trendforce cuts iphone estimate after foxconn ...,5,1,0,7,"[0.18357043, -0.42415977, 0.5561008, 0.2305212..."


# **Rename embedding to bert_embedding and download new dataset**

In [ ]:
training.rename(columns={'embedding': 'bert_embedding'}, inplace=True)

training.to_csv('training_embed.csv', index=False)

from google.colab import files
files.download('training_embed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **FinancialBERT (0.1B Parameters) Embedding**

In [ ]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("ahmedrachid/FinancialBERT")
model     = BertModel.from_pretrained("ahmedrachid/FinancialBERT")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Same as BERT Implementation
def get_financialbert_embedding(text):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors='pt',
        add_special_tokens=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state
    mean_pooled      = token_embeddings.mean(dim=1)
    return mean_pooled.squeeze().numpy()

Some weights of BertModel were not initialized from the model checkpoint at ahmedrachid/FinancialBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Apply the embedding function row by row
training['embedding'] = training['clean_text'].apply(get_financialbert_embedding)

In [ ]:
training.rename(columns={'embedding': 'financialBERT_embedding'}, inplace=True)
training.head(20)

training.to_csv('training_embed.csv', index=False)

from google.colab import files
files.download('training_embed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Llama 3.1 (8B Parameters) Embedding**

In [ ]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install transformers sentence-transformers

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5" # Well-performing and fast default from Hugging Face
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Generates a 1D embedding vector of 384 dimensions
def get_llama_embedding(text):
    return Settings.embed_model.get_text_embedding(text)

In [ ]:
# Apply the embedding function row by row
training['llama_embedding'] = training['clean_text'].apply(get_llama_embedding)

In [ ]:
training.head(20)

training.to_csv('training_embed.csv', index=False)

from google.colab import files
files.download('training_embed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Embeddings on Validation Set**

In [ ]:
# BERT Embedding
valid['bert_embedding'] = valid['clean_text'].apply(get_bert_embedding)

In [ ]:
# FinancialBERT Embedding
valid['financialBERT_embedding'] = valid['clean_text'].apply(get_financialbert_embedding)

In [ ]:
# Llama 3.1 Embedding
valid['llama_embedding'] = valid['clean_text'].apply(get_llama_embedding)

In [ ]:
valid.head(20)

valid.to_csv('valid_embed.csv', index=False)

from google.colab import files
files.download('valid_embed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import csv
[;l.]

combined.to_csv("original_twitter_embed.csv", index=False)

# Class Balancing For Training Dataset

In [ ]:
# Issue is that labels 2 has a much greater sample size (64.7% of dataset) compared to 1 (20%) and 0 (15%)
training["label"].value_counts(normalize=True)

Neutral class comprises of 64.7% of the dataset compared to Bearish (15%) and Bullish (20%)

In [ ]:
import pandas as pd

# Class balance the training dataset while leaving our test csv alone
training_embed = pd.read_csv("training_embed.csv")

df_class0 = training_embed[training_embed['label'] == 0]
df_class1 = training_embed[training_embed['label'] == 1]
df_class2 = training_embed[training_embed['label'] == 2]

# sample without replacement the dataframe that contains class 2 for only 2000
df_class2_resample = df_class2.sample(n=2000, replace=False, random_state=42)

# stich the rows back together
df_balanced = pd.concat([df_class0, df_class1, df_class2_resample], axis=0)
print(df_balanced['label'].value_counts())
print(training_embed['label'].value_counts())

df_balanced.to_csv('balanced_training_embed.csv', index=False)




label
2    2000
1    1923
0    1442
Name: count, dtype: int64
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


The original training dataset had 6178 for Neutral, 1923 for Bullish, and 1442 for Bearish.

With the newly balanced training dataset, we have 2000 for Neutral, 1923 for Bullish, and 1442 for Bearish.